# Concurrent Futures in Python (ThreadPool & ProcessPool)

This notebook is a **from-scratch to intermediate/advanced** guide to Python's
`concurrent.futures` module: **Futures**, **ThreadPoolExecutor**, and **ProcessPoolExecutor**.

We focus on:

- **I/O-bound work** → **threads** (e.g. many HTTP requests in parallel)
- **CPU-bound work** → **processes** (e.g. heavy numerical calculations)

We use **timing and delays** throughout to compare sequential vs concurrent execution
and to illustrate when each approach pays off.

Outline:

1. **Core ideas**: Future, Executor, submit, and map
2. **Baseline**: sequential HTTP and CPU tasks with timing
3. **ThreadPoolExecutor** for I/O-bound HTTP requests (with timing comparison)
4. **ProcessPoolExecutor** for CPU-bound calculations (with timing comparison)
5. **as_completed**, **wait**, and exception handling
6. **Timeouts and cancellation**
7. **Best practices and pitfalls**

> Run the code cells and compare the printed timings. Adjust delay/worker count to see
> how behavior changes.

## 1. Core Ideas: Future, Executor, submit, and map

- **Executor**: abstract interface to run callables in a pool of workers.
  - `ThreadPoolExecutor` → uses threads (shared memory, good for I/O).
  - `ProcessPoolExecutor` → uses separate processes (no GIL, good for CPU).
- **Future**: a handle to a result that may not be ready yet. You can `.result()` to block
  until done, or check `.done()`, `.cancelled()`, and handle exceptions.
- **submit(fn, *args, **kwargs)**: schedule one task; returns a `Future`.
- **map(fn, *iterables)**: schedule many tasks and iterate over results in order.

We'll use **time.perf_counter()** to measure wall-clock time and compare runs.

In [1]:
# 1.1 Minimal Future and submit example

import time
from concurrent.futures import ThreadPoolExecutor, Future


def slow_task(name: str, delay: float) -> str:
    """Simulate an I/O-bound task (e.g. HTTP request) with a sleep."""
    time.sleep(delay)
    return f"done:{name}"


# Context manager ensures executor shuts down and waits for workers
with ThreadPoolExecutor(max_workers=2) as executor:
    # submit() returns a Future immediately; work runs in a worker thread
    f1: Future[str] = executor.submit(slow_task, "A", 0.5)
    f2: Future[str] = executor.submit(slow_task, "B", 0.3)

    # .result() blocks until the task completes and returns the value
    print(f1.result())
    print(f2.result())

done:A
done:B


## 2. Baseline: Sequential Execution with Timing

Before parallelizing, we establish **sequential baselines** for:

- **I/O-bound**: simulate N HTTP requests (each takes ~0.2 s delay).
- **CPU-bound**: run N heavy calculations (e.g. compute-intensive function).

Total sequential time ≈ N × time_per_task. We'll then compare with thread/process pools.

In [2]:
# 2.1 Sequential I/O-bound: simulated HTTP requests

import time

# Simulate one HTTP request (e.g. to a REST API) with a fixed delay
REQUEST_DELAY = 0.2
NUM_REQUESTS = 5


def simulated_http_get(url_id: int) -> dict:
    """Simulate an HTTP GET: wait REQUEST_DELAY seconds, then return a fake response."""
    time.sleep(REQUEST_DELAY)
    return {"url_id": url_id, "status": 200, "data": f"response_{url_id}"}


# Sequential run: one request after another
urls = list(range(NUM_REQUESTS))
t0 = time.perf_counter()
results_seq = [simulated_http_get(i) for i in urls]
elapsed_seq = time.perf_counter() - t0

print(f"Sequential: {NUM_REQUESTS} requests, {REQUEST_DELAY}s each")
print(f"Total time: {elapsed_seq:.3f}s (expected ~{NUM_REQUESTS * REQUEST_DELAY:.3f}s)")
print("Results:", [r["url_id"] for r in results_seq])

Sequential: 5 requests, 0.2s each
Total time: 1.002s (expected ~1.000s)
Results: [0, 1, 2, 3, 4]


In [3]:
# 2.2 Sequential CPU-bound: heavy calculation

def cpu_heavy(n: int, iterations: int = 5_000_000) -> float:
    """Burn CPU for a measurable time (e.g. simulate a complex pricing/risk calculation)."""
    total = 0.0
    for i in range(iterations):
        total += (i * 0.0001) ** 0.5
    return total + n


NUM_CPU_TASKS = 4

# Run all CPU tasks one after another; total time ≈ sum of each task
t0 = time.perf_counter()
results_cpu_seq = [cpu_heavy(i) for i in range(NUM_CPU_TASKS)]
elapsed_cpu_seq = time.perf_counter() - t0

print(f"Sequential CPU: {NUM_CPU_TASKS} tasks")
print(f"Total time: {elapsed_cpu_seq:.3f}s")
print("Results (first 2):", results_cpu_seq[:2])

Sequential CPU: 4 tasks
Total time: 1.044s
Results (first 2): [74535588.06757535, 74535589.06757535]


## 3. ThreadPoolExecutor for I/O-Bound Work (HTTP Requests)

For **I/O-bound** tasks (network, disk), threads work well: while one thread waits on I/O,
others can run. The GIL is released during I/O, so you get real concurrency.

We run the same N "HTTP" requests using **ThreadPoolExecutor** and compare total time.
With enough workers, total time ≈ one request time (plus overhead).

In [2]:
# 3.1 ThreadPoolExecutor: parallel simulated HTTP requests

from concurrent.futures import ThreadPoolExecutor

MAX_WORKERS = 5

# All requests run concurrently; total time ≈ one request + overhead
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(simulated_http_get, i) for i in urls]
    results_thread = [f.result() for f in futures]  # collect in submission order
elapsed_thread = time.perf_counter() - t0

print(f"ThreadPool: {NUM_REQUESTS} requests, max_workers={MAX_WORKERS}")
print(f"Total time: {elapsed_thread:.3f}s")
print(f"Speedup vs sequential: {elapsed_seq / elapsed_thread:.2f}x")
print("Results:", [r["url_id"] for r in results_thread])

NameError: name 'time' is not defined

In [5]:
# 3.2 Same I/O workload with executor.map (results in order)

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # map preserves order of iterable; blocks until all complete
    results_map = list(executor.map(simulated_http_get, urls))
elapsed_map = time.perf_counter() - t0

print(f"ThreadPool map: {elapsed_map:.3f}s")
print("Results (order preserved):", [r["url_id"] for r in results_map])

ThreadPool map: 0.202s
Results (order preserved): [0, 1, 2, 3, 4]


## 4. ProcessPoolExecutor for CPU-Bound Work

For **CPU-bound** work, threads are limited by the **GIL** (Global Interpreter Lock).
**ProcessPoolExecutor** runs tasks in **separate processes**, each with its own interpreter,
so you get true parallel CPU use.

Trade-offs: process startup and IPC have overhead; use when each task does enough
work to justify it. We run the same CPU-heavy tasks in a process pool and compare.

In [6]:
# 4.1 ProcessPoolExecutor: parallel CPU-heavy tasks

from concurrent.futures import ProcessPoolExecutor

# Reuse cpu_heavy from section 2.2 (must be top-level for pickling in process pool)

# Each task runs in a separate process; no GIL, so real parallel CPU use
t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=NUM_CPU_TASKS) as executor:
    futures = [executor.submit(cpu_heavy, i) for i in range(NUM_CPU_TASKS)]
    results_cpu_par = [f.result() for f in futures]
elapsed_cpu_par = time.perf_counter() - t0

print(f"ProcessPool: {NUM_CPU_TASKS} CPU tasks, max_workers={NUM_CPU_TASKS}")
print(f"Total time: {elapsed_cpu_par:.3f}s (sequential was {elapsed_cpu_seq:.3f}s)")
print(f"Speedup vs sequential: {elapsed_cpu_seq / elapsed_cpu_par:.2f}x")
print("Results (first 2):", results_cpu_par[:2])

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [7]:
# 4.2 ProcessPool with map (order preserved)

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=NUM_CPU_TASKS) as executor:
    results_cpu_map = list(executor.map(cpu_heavy, range(NUM_CPU_TASKS)))
elapsed_cpu_map = time.perf_counter() - t0

print(f"ProcessPool map: {elapsed_cpu_map:.3f}s")
print("Results:", results_cpu_map[:3])

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

### 4.3 Note on ProcessPool and Jupyter

In **Jupyter/IPython**, the process pool runs in child processes that import your
notebook as `__main__`. Top-level functions (e.g. `cpu_heavy`, `simulated_http_get`)
defined in earlier cells are picklable. If you move this code to a `.py` module,
define worker functions at module level so they can be pickled.

## 5. as_completed, wait, and Exception Handling

- **as_completed(futures, timeout=None)**: yields futures as they complete (order not
  guaranteed). Useful when you want to process results as soon as they're ready.
- **wait(futures, return_when=ALL_COMPLETED)**: block until a condition is met; returns
  a named tuple with `done` and `not_done` sets.
- **Future.result(timeout)** raises `TimeoutError` if not ready in time; **Future.exception()**
  returns the exception raised by the task (if any) instead of re-raising.

We'll use **as_completed** to process HTTP results as they arrive, and show how to
collect exceptions from futures.

In [ ]:
# 5.1 Processing results as they complete with as_completed

from concurrent.futures import as_completed

def simulated_http_get_with_delay(url_id: int, delay: float) -> dict:
    """Same as before but delay is per-call for variety."""
    time.sleep(delay)
    return {"url_id": url_id, "status": 200}

# Different delays so completions are out of order
delays = [0.3, 0.1, 0.2, 0.25, 0.15]

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {
        executor.submit(simulated_http_get_with_delay, i, d): i
        for i, d in enumerate(delays)
    }
    for future in as_completed(futures):
        url_id = futures[future]
        try:
            result = future.result()
            print(f"  Completed url_id={url_id} at {time.perf_counter() - t0:.2f}s")
        except Exception as e:
            print(f"  url_id={url_id} failed: {e}")
print(f"Total: {time.perf_counter() - t0:.3f}s")

In [ ]:
# 5.2 wait() and handling exceptions from futures

from concurrent.futures import wait, FIRST_EXCEPTION

def sometimes_fails(url_id: int) -> str:
    time.sleep(0.1)
    if url_id == 2:
        raise ValueError("Simulated failure for url_id=2")
    return f"ok_{url_id}"

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(sometimes_fails, i) for i in range(5)]
    done, not_done = wait(futures, return_when=FIRST_EXCEPTION)
    # Process completed; check for exceptions
    for f in done:
        if f.exception() is not None:
            print("Task raised:", f.exception())
        else:
            print("Result:", f.result())
    print("Not done yet:", len(not_done))

## 6. Timeouts and Cancellation

- **future.result(timeout=T)**: block at most T seconds; raises `TimeoutError` if not done.
- **future.cancel()**: try to cancel the task; returns True only if it was not yet running.
- **executor.map(..., timeout=T)**: same timeout applies when iterating results.

We'll submit a long-running task and call **result(timeout=...)** to avoid blocking forever.

In [ ]:
# 6.1 result(timeout=...) and cancel()

def long_task(sec: float) -> str:
    time.sleep(sec)
    return "done"

with ThreadPoolExecutor(max_workers=2) as executor:
    f_slow = executor.submit(long_task, 2.0)
    f_fast = executor.submit(long_task, 0.2)

    print("Fast result:", f_fast.result(timeout=1.0))

    try:
        # Wait at most 0.5s for the slow task
        print("Slow result:", f_slow.result(timeout=0.5))
    except TimeoutError:
        print("Slow task did not finish within 0.5s (expected)")

    # Try to cancel (may already be running)
    cancelled = f_slow.cancel()
    print("Cancel returned:", cancelled)

## 7. Summary: When to Use Which

| Workload   | Sequential | ThreadPoolExecutor      | ProcessPoolExecutor   |
|-----------|------------|-------------------------|------------------------|
| I/O-bound | N × delay  | ~delay (with N workers) | Overkill, more overhead |
| CPU-bound | N × cpu    | Limited by GIL          | ~N-way speedup (real cores) |

- Use **threads** for many independent I/O operations (HTTP, DB, file).
- Use **processes** for CPU-heavy, parallelizable calculations.
- Always **measure** with realistic data sizes; process startup and IPC have cost.
- Prefer **context manager** `with Executor(...) as ex:` so shutdown is clean.

## 8. Best Practices and Pitfalls

### Do

- Use **with** so the executor shuts down and waits for workers.
- Prefer **submit** + **as_completed** when order doesn't matter and you want to react
  to the first completions; use **map** when you need results in order.
- Set **timeout** on `.result()` or in `map` to avoid hanging on stuck tasks.
- For process pools, keep **worker functions** and their arguments **picklable** (top-level
  functions, built-in types, dataclasses, etc.).

### Avoid

- Passing **lambda** or **nested functions** to ProcessPoolExecutor (pickling issues).
- Using **ThreadPoolExecutor** for CPU-bound work (GIL limits gains).
- Starting huge numbers of workers; match **max_workers** to I/O concurrency or CPU cores.
- Forgetting to call **.result()** on futures (or iterating map) so exceptions are raised
  and not silently ignored.

### Optional: Real HTTP requests with ThreadPoolExecutor

If you have network access, you can run the same pattern against real endpoints (e.g.
`https://httpbin.org/delay/1`). Replace `simulated_http_get` with a function that
uses `urllib.request` or `requests`; the executor usage stays the same. Measure wall-clock
time to see the benefit of parallel I/O.

In [ ]:
# Optional: real HTTP with ThreadPoolExecutor (requires network)

import urllib.request
import urllib.error

def real_http_get(url: str, timeout: float = 5.0) -> dict:
    """Perform a real GET request; returns status and length or error."""
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            body = resp.read()
            return {"url": url, "status": resp.status, "length": len(body)}
    except Exception as e:
        return {"url": url, "error": str(e)}

# Example: hit httpbin.org delay endpoints (1s each); parallel = ~1s total
urls_real = [
    "https://httpbin.org/delay/1",
    "https://httpbin.org/delay/1",
    "https://httpbin.org/delay/1",
]

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=3) as executor:
    results_real = list(executor.map(real_http_get, urls_real))
elapsed_real = time.perf_counter() - t0

print(f"Real HTTP: 3 x 1s delay URLs -> {elapsed_real:.2f}s total (sequential would be ~3s)")
for r in results_real:
    print(" ", r)